# 17.8 Actor-Critic 与 A2C / Actor-Critic & A2C

**中文**：REINFORCE(17.7)有两个痛点:①必须**等一整局结束**才能更新(用蒙特卡洛回报);②即使减了基线,方差还是大。本节的 **Actor-Critic(演员-评论家)** 优雅地同时解决——它把**策略网络(actor,演员:负责做动作)** 和 **价值网络(critic,评论家:负责评价动作好坏)** 合在一起。评论家提供一个**会自举、边学边更新的基线**,让我们**每几步就能更新一次**,不必等回合结束。Actor-Critic 是 **A2C/A3C/PPO/SAC** 等一切现代 RL 算法的通用骨架。
**English**: REINFORCE (17.7) has two pain points: ① it must **wait for a whole episode** to update (Monte-Carlo returns); ② even with a baseline, variance stays high. **Actor-Critic** elegantly solves both — it unites a **policy network (actor: takes actions)** and a **value network (critic: judges how good actions are)**. The critic provides a **bootstrapping, continuously-updated baseline**, letting us **update every few steps** without waiting for the episode to end. Actor-Critic is the universal skeleton of all modern RL (A2C/A3C/PPO/SAC).

---

**中文**：核心是把 REINFORCE 里"用整回合真实回报 $G_t$"换成"评论家的**自举估计**"。**优势(advantage)** 有多种算法,它们构成一个**偏差-方差的连续谱**:
**English**: The core is replacing REINFORCE's "full-episode real return $G_t$" with the critic's **bootstrapped estimate**. The **advantage** can be computed several ways, forming a **bias-variance spectrum**:

$$\underbrace{A_t=r_t+\gamma V(s_{t+1})-V(s_t)}_{\text{1步TD:低方差,高偏差}}\quad\cdots\quad\underbrace{A_t=\sum_{k=0}^{n-1}\gamma^k r_{t+k}+\gamma^n V(s_{t+n})-V(s_t)}_{\text{n步}}\quad\cdots\quad\underbrace{A_t=G_t-V(s_t)}_{\text{MC:无偏,高方差}}$$

**中文**：
- **1 步 TD**(只看 1 步真实奖励 + 评论家估计):方差最小,但**严重依赖评论家准不准**——评论家不准就偏差大。
- **蒙特卡洛(MC)**(用整回合真实回报):无偏,但方差最大(REINFORCE 就是这端)。
- **n 步**:在两者之间取平衡。**GAE(广义优势估计)** 用一个参数 $\lambda$ 平滑地在整条谱上插值——这正是 PPO(下节)用的。

**English**:
- **1-step TD** (one real reward + critic estimate): lowest variance, but **heavily depends on the critic's accuracy** — a bad critic means high bias.
- **Monte Carlo (MC)** (full-episode real return): unbiased, highest variance (this is REINFORCE's end).
- **n-step**: balances the two. **GAE (Generalized Advantage Estimation)** smoothly interpolates the whole spectrum with a parameter $\lambda$ — exactly what PPO (next) uses.

**中文**：训练时两个损失同时优化:**演员**用策略梯度 $-\log\pi(a|s)\cdot A$(A 用评论家算)、**评论家**用 TD/回归让 $V(s)$ 逼近真实回报。**A2C** = 同步版本(多个并行环境采样后一起更新);**A3C** = 异步版本(多个 worker 各自异步更新)。
**English**: Two losses are optimized jointly: the **actor** via policy gradient $-\log\pi(a|s)\cdot A$ (A from the critic), the **critic** via TD/regression to make $V(s)$ approach the real return. **A2C** = synchronous (parallel environments sampled then updated together); **A3C** = asynchronous (multiple workers update asynchronously).

> 💡 **面试速查 / Interview cheat-sheet（★★★ 现代 RL 骨架必考）**
> **中文**：**Actor-Critic=策略(actor)+价值(critic)**。评论家=**会自举的学习型基线**→比 REINFORCE 方差更低、**能每步在线更新**(不等回合结束, 支持连续任务)。**优势估计是偏差-方差旋钮**:1步TD(低方差高偏差, 依赖 critic)↔ MC(无偏高方差), n步/**GAE(λ)** 插值。两损失:actor 用 $-\log\pi\cdot A$, critic 用 $\|V-\text{目标}\|^2$; 常加**熵正则**促探索。**A2C**同步/**A3C**异步。这是 PPO、SAC、RLHF 的共同底座。
> **English**: **Actor-Critic = policy (actor) + value (critic)**. The critic is a **bootstrapping learned baseline** → lower variance than REINFORCE and **online per-step updates** (no waiting for episode end, supports continuing tasks). **The advantage estimator is a bias-variance knob**: 1-step TD (low variance, high bias, critic-dependent) ↔ MC (unbiased, high variance), with n-step / **GAE(λ)** interpolating. Two losses: actor $-\log\pi\cdot A$, critic $\|V-\text{target}\|^2$; often add an **entropy bonus** for exploration. **A2C** synchronous / **A3C** asynchronous. This is the shared base of PPO, SAC, and RLHF.


In [ ]:

# ============================================================
# 环境 + Actor-Critic 网络(共享主干)/ CartPole + shared-trunk Actor-Critic net
# ============================================================
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, random, time, matplotlib.pyplot as plt
def set_seed(x): torch.manual_seed(x); np.random.seed(x); random.seed(x)
class CartPole:
    g=9.8; mc=1.0; mp=0.1; l=0.5; fm=10.0; tau=0.02
    def reset(s): s.state=np.random.uniform(-0.05,0.05,4); s.steps=0; return s.state.copy()
    def step(s,a):
        x,xd,th,thd=s.state; force=s.fm if a==1 else -s.fm
        ct,st=np.cos(th),np.sin(th); tot=s.mc+s.mp
        temp=(force+s.mp*s.l*thd**2*st)/tot
        thacc=(s.g*st-ct*temp)/(s.l*(4/3-s.mp*ct**2/tot)); xacc=temp-s.mp*s.l*thacc*ct/tot
        x+=s.tau*xd; xd+=s.tau*xacc; th+=s.tau*thd; thd+=s.tau*thacc
        s.state=np.array([x,xd,th,thd]); s.steps+=1
        done=abs(x)>2.4 or abs(th)>12*np.pi/180 or s.steps>=500
        return s.state.copy(),1.0,done

class ActorCritic(nn.Module):
    def __init__(s):
        super().__init__()
        s.trunk=nn.Sequential(nn.Linear(4,128),nn.ReLU())    # 共享特征主干 / shared trunk
        s.actor=nn.Linear(128,2)                             # 演员头:动作概率 / policy head
        s.critic=nn.Linear(128,1)                            # 评论家头:状态价值 V(s) / value head
    def forward(s,x):
        h=s.trunk(x); return F.softmax(s.actor(h),dim=-1), s.critic(h).squeeze(-1)
print("Actor-Critic(演员+评论家, 共享主干)就绪 / ready")


**中文**：从零实现 **n 步 Actor-Critic**:每采集 $n$ 步就更新一次(而非等整回合)。用 $n$ 步真实奖励 + 第 $n$ 步的评论家自举估计算回报，得到优势;演员用优势做策略梯度，评论家回归到该回报。我们把 $n$ 做成参数，好直接对比"1 步 TD / n 步 / 蒙特卡洛"三种优势估计。
**English**: Implement **n-step Actor-Critic** from scratch: update every $n$ steps (not per episode). Compute returns from $n$ real rewards + the critic's bootstrap at step $n$, forming the advantage; the actor does policy gradient with it, the critic regresses to that return. We make $n$ a parameter to directly compare "1-step TD / n-step / Monte Carlo" advantage estimation.


In [ ]:

# ============================================================
# n 步 Actor-Critic / n-step Actor-Critic (n=1: TD, n large: MC)
# ============================================================
def train_ac(nstep, episodes=450, gamma=0.99, ent_coef=0.01, seed=0):
    set_seed(seed); env=CartPole(); net=ActorCritic(); opt=torch.optim.Adam(net.parameters(), lr=3e-3)
    lengths=[]
    for ep in range(episodes):
        s=env.reset(); done=False; total=0
        while not done:
            logps=[]; vals=[]; rews=[]; ents=[]
            for _ in range(nstep):                            # 采集最多 n 步 / collect up to n steps
                st=torch.tensor(s,dtype=torch.float32); probs,v=net(st)
                dist=torch.distributions.Categorical(probs); a=dist.sample()
                logps.append(dist.log_prob(a)); vals.append(v); ents.append(dist.entropy())
                s,r,done=env.step(int(a)); rews.append(r); total+=1
                if done: break
            with torch.no_grad():                             # 自举:末状态的评论家估计 / bootstrap
                R = 0.0 if done else float(net(torch.tensor(s,dtype=torch.float32))[1])
            rets=[]
            for r in reversed(rews): R=r+gamma*R; rets.append(R)   # n 步折扣回报 / n-step returns
            rets=torch.tensor(rets[::-1],dtype=torch.float32); V=torch.stack(vals)
            adv=(rets - V).detach()                           # 优势 = 回报 - 评论家 / advantage
            actor_loss=-(torch.stack(logps)*adv).sum()        # 演员:策略梯度 / policy gradient
            critic_loss=F.mse_loss(V, rets)                   # 评论家:回归到回报 / value regression
            ent=torch.stack(ents).sum()
            loss=actor_loss + 0.5*critic_loss - ent_coef*ent  # 合并 + 熵正则 / + entropy bonus
            opt.zero_grad(); loss.backward(); opt.step()
        lengths.append(total)
    return lengths

t=time.time()
td1  = train_ac(nstep=1)      # 纯 1 步 TD(极度依赖评论家)/ pure 1-step TD
nstep= train_ac(nstep=16)     # n 步(平衡)/ n-step (balanced)
mc   = train_ac(nstep=10000)  # 蒙特卡洛(整回合)/ Monte Carlo (whole episode)
print(f"{'优势估计/advantage':<24}{'最后50回合平均存活':>18}")
print(f"{'1步 TD (n=1)':<24}{np.mean(td1[-50:]):>18.0f}")
print(f"{'n步 (n=16)':<24}{np.mean(nstep[-50:]):>18.0f}")
print(f"{'蒙特卡洛 MC':<24}{np.mean(mc[-50:]):>18.0f}")
print(f"用时 {time.time()-t:.0f}s | 满分500 随机~20")


**中文**：结果揭示了一个**诚实且重要**的现象:**纯 1 步 TD 反而学不起来**! 下面可视化三种优势估计的学习曲线，并解释为什么。
**English**: The result reveals an **honest and important** phenomenon: **pure 1-step TD actually fails to learn**! Let's visualize the three advantage estimators' learning curves and explain why.


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
def smooth(x,k=15): return np.convolve(x,np.ones(k)/k,mode="valid")
fig,ax=plt.subplots(1,2,figsize=(14,4.7))
ax[0].plot(smooth(td1),color="#C44E52",lw=2,label=f"1步 TD ({np.mean(td1[-50:]):.0f})")
ax[0].plot(smooth(nstep),color="#4C72B0",lw=2,label=f"n步 n=16 ({np.mean(nstep[-50:]):.0f})")
ax[0].plot(smooth(mc),color="#55A868",lw=2,label=f"MC ({np.mean(mc[-50:]):.0f})")
ax[0].axhline(500,ls=":",color="gray"); ax[0].set_title("三种优势估计 / three advantage estimators")
ax[0].set_xlabel("episode"); ax[0].set_ylabel("存活步数 survival"); ax[0].legend(fontsize=9)
# 偏差-方差谱示意 / the bias-variance spectrum schematic
ax[1].axis("off")
ax[1].text(0.5,0.92,"优势估计的偏差-方差谱\nbias-variance of advantage",ha="center",fontsize=12,weight="bold",transform=ax[1].transAxes)
ax[1].annotate("", xy=(0.95,0.5), xytext=(0.05,0.5), arrowprops=dict(arrowstyle="<->",lw=2), transform=ax[1].transAxes)
ax[1].text(0.05,0.62,"1步 TD\n低方差\n高偏差(依赖critic)",ha="left",fontsize=9,color="#C44E52",transform=ax[1].transAxes)
ax[1].text(0.5,0.62,"n步 / GAE\n(平衡, 实战首选)",ha="center",fontsize=9,color="#4C72B0",transform=ax[1].transAxes)
ax[1].text(0.95,0.62,"MC\n无偏\n高方差",ha="right",fontsize=9,color="#55A868",transform=ax[1].transAxes)
ax[1].text(0.5,0.25,"PPO 用 GAE(λ) 在这条谱上\n平滑插值取甜点\nPPO uses GAE(λ) to interpolate",ha="center",fontsize=10,transform=ax[1].transAxes)
plt.tight_layout(); plt.savefig("/tmp/rl08_viz.png",dpi=80); plt.show()
print("1步TD失败, n步/MC成功 —— 自举的偏差在评论家不准时会毒害学习")


**中文**：诚实解读:
**English**: Honest takeaways:

**中文**：
1. **诚实的反直觉:纯 1 步 TD 反而失败,中间的 n 步才是赢家**。理论上 1 步 TD 方差最小，应该好训。但它的优势 $A=r+\gamma V(s')-V(s)$ **完全依赖评论家 $V$ 准不准**——训练早期评论家一塌糊涂，这个高偏差的信号会把演员带偏，恶性循环，学不起来(~9 步)。另一极端的**纯 MC 虽无偏但方差太大**、学得又慢又抖(~125)。**恰恰是中间的 n 步(n=16)稳稳夺冠(~457,逼近满分)**——它用适量真实奖励压低方差、又用一点自举补足,正好落在偏差-方差的甜点上。**这戳破了"自举总是更好"和"无偏总是更好"两个错觉**。
2. **优势估计是一个偏差-方差旋钮**:从 1 步 TD(低方差高偏差)到 MC(无偏高方差),中间的 n 步是实战甜点。**GAE(λ)** 就是把这条谱用一个 λ 平滑参数化——这正是下一节 PPO 的关键组件。**没有免费的午餐,关键是找到平衡点。**
3. **Actor-Critic 的真正价值不只在"CartPole 分更高"**,更在**架构与能力**:①评论家让你能**每 n 步在线更新**(不等回合结束)→ 支持**连续不终止**的任务(MC 做不到);②它是并行化(A2C/A3C)、信任域(PPO)、连续控制(SAC)的**统一底座**;③在长回合、稀疏奖励的真实任务上,自举带来的低方差与在线更新优势才真正决定性(CartPole 只是小小演示)。

**English**:
1. **An honest counterintuition: pure 1-step TD fails, and the middle n-step is the winner.** In theory 1-step TD has the lowest variance and should train well. But its advantage $A=r+\gamma V(s')-V(s)$ **depends entirely on the critic $V$'s accuracy** — early on the critic is terrible, so this high-bias signal misleads the actor in a vicious cycle, failing to learn (~9 steps). The other extreme, **pure MC, is unbiased but too high-variance**, learning slowly and jitterily (~125). **It is precisely the middle n-step (n=16) that wins decisively (~457, near max)** — using enough real rewards to cut variance plus a bit of bootstrapping, landing right on the bias-variance sweet spot. **This punctures both myths: "bootstrapping is always better" and "unbiased is always better."**
2. **Advantage estimation is a bias-variance knob**: from 1-step TD (low variance, high bias) to MC (unbiased, high variance), with n-step as the practical sweet spot in between. **GAE(λ)** parameterizes this whole spectrum smoothly with one λ — the key component of PPO next. **No free lunch; the art is finding the balance.**
3. **Actor-Critic's real value is not only "higher CartPole score"** but its **architecture and capabilities**: ① the critic lets you **update online every n steps** (no waiting for episode end) → supports **continuing (non-terminating)** tasks (MC cannot); ② it is the **unified base** for parallelization (A2C/A3C), trust regions (PPO), and continuous control (SAC); ③ on long-horizon, sparse-reward real tasks, bootstrapping's low variance and online updates become decisive (CartPole is just a small demo).

> 💼 **实战视角 / Practical angle**
> **中文**:Actor-Critic 是现代 RL 的**事实标准骨架**。工程要点:①**几乎不用纯 1 步**,而用 **n 步或 GAE(λ≈0.95)** 平衡偏差方差;②演员/评论家可共享主干(省参)或分开(更稳);③**熵正则**几乎必加(防策略过早收敛、鼓励探索);④critic 的学习率/权重要调好(critic 崩了 actor 也跟着崩)。**A2C(同步)** 比 **A3C(异步)** 更常用(实现简单、GPU 友好)。面试金句:*"Actor-Critic 用会自举的评论家当基线, 兼顾 REINFORCE 的直接优化和 TD 的低方差/在线更新; 但自举有偏差, 所以实战用 n 步/GAE 在偏差-方差谱上取甜点——这就是 PPO 的基础。"*
> **English**: Actor-Critic is the **de facto skeleton** of modern RL. Engineering: ① **almost never pure 1-step**; use **n-step or GAE (λ≈0.95)** to balance bias-variance; ② actor/critic can share a trunk (fewer params) or be separate (more stable); ③ an **entropy bonus** is nearly always added (prevent premature policy collapse, encourage exploration); ④ tune the critic's learning rate/weight (if the critic collapses, so does the actor). **A2C (synchronous)** is more common than **A3C (asynchronous)** (simpler, GPU-friendly). Interview line: *"Actor-Critic uses a bootstrapping critic as the baseline, combining REINFORCE's direct optimization with TD's low variance and online updates; but bootstrapping is biased, so practice uses n-step/GAE to hit the sweet spot on the bias-variance spectrum — the foundation of PPO."*

---
### 小结 / Summary
- **中文**:Actor-Critic=演员(策略)+评论家(价值); 评论家提供会自举的学习型基线, 支持每步在线更新。
- **English**: Actor-Critic = actor (policy) + critic (value); the critic provides a bootstrapping learned baseline, enabling online per-step updates.
- **中文**:优势估计是偏差-方差旋钮(1步TD↔n步↔MC); 诚实结果:纯1步TD因 critic 偏差反而失败, n步/GAE 是甜点。
- **English**: Advantage estimation is a bias-variance knob (1-step TD ↔ n-step ↔ MC); honestly, pure 1-step TD fails from critic bias, n-step/GAE is the sweet spot.
- **中文**:AC 是 A2C/A3C/PPO/SAC 的统一骨架, 真正价值在架构与在线/连续任务能力, 引出 PPO。
- **English**: AC is the unified skeleton of A2C/A3C/PPO/SAC; its real value is the architecture and online/continuing-task capability — motivating PPO.
